# Tutorial 1 — Project Setup

The `Project` object is the single source of truth for every path in the pipeline.
Nothing is hard-coded inside the package; every directory is either supplied by the
caller or derived from a common `root`.

This tutorial covers:

- Constructing a `Project` from keyword arguments
- Loading from a YAML file
- Loading from an environment variable
- Inspecting the derived directories
- The `contract()` method and what it checks

In [1]:
from pathlib import Path
import tempfile, os

import scads_drvi as sd
from scads_drvi.config import Project

## 1. Construct from keyword arguments

`fit` and `traits` have no defaults — guessing either silently points the pipeline at
the wrong inputs.

In [2]:
root = Path(tempfile.mkdtemp())

proj = Project(
    root=root,
    fit="assembly_v3",
    traits=("plant_height", "grain_yield"),
)
proj

Project(root=PosixPath('/tmp/tmpjhcpw07z'), data=PosixPath('/tmp/tmpjhcpw07z/data'), fits=PosixPath('/tmp/tmpjhcpw07z/data/factorize'), enrich=PosixPath('/tmp/tmpjhcpw07z/data/enrich'), annotations=PosixPath('/tmp/tmpjhcpw07z/data/annotations'), figures=PosixPath('/tmp/tmpjhcpw07z/results/figures'), ldsc_ref=None, ldsc_bin=None, fit='assembly_v3', traits=('plant_height', 'grain_yield'))

### Default directory layout

Each field defaults to a conventional location under `root`. Any of them can be
overridden independently.

In [3]:
print("data:       ", proj.data)
print("fits:       ", proj.fits)
print("enrich:     ", proj.enrich)
print("annotations:", proj.annotations)
print("figures:    ", proj.figures)

data:        /tmp/tmpjhcpw07z/data
fits:        /tmp/tmpjhcpw07z/data/factorize
enrich:      /tmp/tmpjhcpw07z/data/enrich
annotations: /tmp/tmpjhcpw07z/data/annotations
figures:     /tmp/tmpjhcpw07z/results/figures


### Derived path helpers

In [4]:
arm = "assembly_v3_topfrac"

print("fit_dir:    ", proj.fit_dir())
print("enrich_dir: ", proj.enrich_dir(arm))
print("annot_dir:  ", proj.annot_dir(arm))
print("results_dir:", proj.results_dir(arm, "plant_height"))
print("figures_dir:", proj.figures_dir(arm, mkdir=False))

fit_dir:     /tmp/tmpjhcpw07z/data/factorize/assembly_v3
enrich_dir:  /tmp/tmpjhcpw07z/data/enrich/assembly_v3_topfrac
annot_dir:   /tmp/tmpjhcpw07z/data/annotations/assembly_v3_topfrac
results_dir: /tmp/tmpjhcpw07z/data/enrich/assembly_v3_topfrac/results/plant_height
figures_dir: /tmp/tmpjhcpw07z/results/figures/assembly_v3_topfrac


### Override individual directories

`Project` is a frozen dataclass; use `replace()` to derive a modified copy.

In [5]:
proj_custom = proj.replace(figures=Path("/scratch/figs"))
print(proj_custom.figures)

/scratch/figs


## 2. Load from a YAML file

This is the recommended approach for reproducible pipelines: commit the YAML and
pass it to `Project.from_yaml`.

In [6]:
import yaml

yaml_path = root / "project.yaml"
yaml_path.write_text(yaml.dump({
    "root": str(root),
    "fit": "assembly_v3",
    "traits": ["plant_height", "grain_yield"],
}))

proj_from_yaml = Project.from_yaml(yaml_path)
print(proj_from_yaml.fit)
print(proj_from_yaml.traits)

assembly_v3
('plant_height', 'grain_yield')


## 3. Load from environment variables

`from_env()` reads `SCADS_DRVI_ROOT` and accepts keyword overrides for any field.
This is useful for parameterised pipeline scripts.

In [7]:
os.environ["SCADS_DRVI_ROOT"] = str(root)

proj_env = Project.from_env(fit="assembly_v3", traits=("plant_height", "grain_yield"))
print(proj_env.root)

/tmp/tmpjhcpw07z


## 4. The `contract()` method

`project.contract(model)` returns a dictionary of paths that the factorize and enrich
stages must have written before scoring can proceed. It does not check that they exist —
use `io.contract.check_contract()` for that.

The keys are:

| key | what it points to |
|---|---|
| `loadings` | `.npz` or `.tsv` of cells × factors loadings |
| `loadings_npz` | preferred `.npz` path |
| `factor_map` | `factor_map.tsv` mapping dim → annotation index |
| `latent_stats` | per-factor statistics from the factorize stage |

In [8]:
contract = proj.contract(arm)
for key, path in contract.items():
    print(f"{key:15s}: {path}")

loadings       : /tmp/tmpjhcpw07z/data/factorize/assembly_v3/topic_loadings.tsv
loadings_npz   : /tmp/tmpjhcpw07z/data/factorize/assembly_v3/topic_loadings.npz
factors        : /tmp/tmpjhcpw07z/data/factorize/assembly_v3/topic_factors.tsv
latent_stats   : /tmp/tmpjhcpw07z/data/factorize/assembly_v3/inspect/latent_stats.tsv
fit_meta       : /tmp/tmpjhcpw07z/data/factorize/assembly_v3/fit.meta.json
factor_map     : /tmp/tmpjhcpw07z/data/enrich/assembly_v3_topfrac/factor_map.tsv
half_map       : /tmp/tmpjhcpw07z/data/enrich/assembly_v3_topfrac/half_map.tsv
annot_stats    : /tmp/tmpjhcpw07z/data/enrich/assembly_v3_topfrac/annot_stats.tsv
results        : /tmp/tmpjhcpw07z/data/enrich/assembly_v3_topfrac/results


## 5. Serialise to a dictionary

`as_dict()` converts every `Path` to a string, making the project configuration
easy to log or pass to a subprocess.

In [9]:
import json

print(json.dumps(proj.as_dict(), indent=2, default=str))

{
  "root": "/tmp/tmpjhcpw07z",
  "data": "/tmp/tmpjhcpw07z/data",
  "fits": "/tmp/tmpjhcpw07z/data/factorize",
  "enrich": "/tmp/tmpjhcpw07z/data/enrich",
  "annotations": "/tmp/tmpjhcpw07z/data/annotations",
  "figures": "/tmp/tmpjhcpw07z/results/figures",
  "ldsc_ref": null,
  "ldsc_bin": null,
  "fit": "assembly_v3",
  "traits": [
    "plant_height",
    "grain_yield"
  ]
}
